# Predicting FPL starts: what the data actually supports

Reproduces the analysis behind `build_spec_minutes_model.md`.

**What this establishes, in order:**

1. Which columns exist in which seasons (and why `defensive_contribution` is a trap)
2. That a pool-wide Brier score is misleading, with measured numbers
3. That ~98% of the error sits in gameweeks where a player's start state *flipped*
4. That recalibrating a naive baseline beats it, using no new information
5. That the result survives leak-free walk-forward validation

Trimmed from the original analysis notebook to sections 1-11 (the base model) -- sections 12-13 covered a cup/European-rotation signal that lives in a separate, private extension to this package. Outputs are stripped; re-run to regenerate the charts.

**Data:** `vaastav/Fantasy-Premier-League`, a community archive. The official FPL
API holds only the current season, so past seasons are not retrievable from it.

**Target:** Python 3.7 compatible. Needs `pandas`, `numpy`, `plotly`, and internet access.

```
pip install pandas numpy plotly
```

In [ ]:
import io
import urllib.request

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "iframe"

BASE = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data"
SEASON = "2025-26"          # last COMPLETED season
BINS = [-0.01, 0.24, 0.49, 0.74, 1.01]
BIN_NAMES = ["0/4", "1/4", "2-3/4", "4/4"]

def fetch(path):
    req = urllib.request.Request(BASE + "/" + path, headers={"User-Agent": "curl"})
    return urllib.request.urlopen(req, timeout=90).read()

pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| numpy", np.__version__, "| plotly", __import__("plotly").__version__)

## 1. Column availability by season

The API's `history_past` reports a 2024/25 `defensive_contribution` total, which
suggests per-gameweek data exists for that season. It does not. Verify rather
than assume.

In [ ]:
seasons = ["2016-17","2017-18","2018-19","2019-20","2020-21","2021-22",
           "2022-23","2023-24","2024-25","2025-26","2026-27"]
rows = []
for s in seasons:
    try:
        cols = set(pd.read_csv(io.BytesIO(fetch(s + "/gws/gw1.csv")), nrows=1).columns)
        rows.append({"season": s,
                     "minutes": "minutes" in cols,
                     "starts": "starts" in cols,
                     "expected_goals": "expected_goals" in cols,
                     "defensive_contribution": "defensive_contribution" in cols})
    except Exception as exc:
        rows.append({"season": s, "minutes": "ERR: %r" % exc})
pd.DataFrame(rows).set_index("season")

`starts` begins in **2022/23**; `defensive_contribution` in **2025/26**.

Anything before those seasons will load without error and contribute zeros.
That is the failure mode to guard against: absent data that looks like observed
zeros. Usable window for a minutes model is 2022/23 onward, and for DefCon just
one complete season plus the current one.

## 2. Load a full season and build walk-forward features

Both features use `.shift(1)` so no row ever sees its own outcome.

In [ ]:
raw = pd.read_csv(io.BytesIO(fetch(SEASON + "/gws/merged_gw.csv")), low_memory=False)
raw = raw.sort_values(["element", "GW"])
raw["y"] = raw["starts"].astype(int)

# previous gameweek's start state
raw["prev"] = raw.groupby("element")["y"].shift(1)
# share of the previous 4 gameweeks started (shifted, so strictly historical)
raw["roll4"] = raw.groupby("element")["y"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean())

d = raw[raw["prev"].notna() & raw["roll4"].notna()].copy()
print("rows %d | gameweeks %d | players %d" % (len(raw), raw.GW.nunique(), raw.element.nunique()))
print("usable rows after feature construction: %d" % len(d))
print("pool-wide base rate of starting: %.3f" % raw.y.mean())

## 3. Strata

Four groups by start rate. Critically, the label is computed **from prior
gameweeks only** — a season-level label leaks the test period into the group
that matters most.

The denominator is gameweeks *registered*, not gameweeks *available*, so an
injured first-choice starter drifts toward Rotation. That is a known limitation.

In [ ]:
def label_from(history):
    """Assign strata using only the gameweeks passed in."""
    g = history.groupby("element")["y"].agg(["sum", "size"])
    rate = g["sum"] / g["size"]
    return pd.Series(
        np.where(g["sum"] == 0, "Deep",
        np.where(rate >= 0.50, "Core",
        np.where(rate >= 0.15, "Rotation", "Marginal"))), index=g.index)

STRATA = ["Core", "Rotation", "Marginal", "Deep"]

# illustrative only: full-season labels, to show the population shape
full = label_from(raw)
g = raw.groupby("element")["y"].agg(["sum", "size"])
summary = pd.DataFrame({"players": full.value_counts()})
summary["start_rate"] = [raw[raw.element.isin(full[full == s].index)].y.mean() for s in summary.index]
summary.loc[STRATA]

## 4. The class imbalance trap

Baseline Brier scores, pool-wide and by stratum.

Brier is mean squared error on probabilities: `mean((p - outcome)**2)`.
Lower is better. Predicting the base rate scores `p*(1-p)`.

In [ ]:
def brier(p, y):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))

ev = d[d.GW >= 6].copy()
ev["stratum"] = ev["element"].map(full)
y = ev["y"].values
base = y.mean()

cands = {
    "constant %.3f (base rate)" % base: np.full(len(ev), base),
    "constant 0.900 (optimistic)": np.full(len(ev), 0.90),
    "persistence (started last GW)": np.clip(ev["prev"].values, 0.05, 0.95),
}
out = {}
for name, p in cands.items():
    row = {"POOL": brier(p, y)}
    for s in STRATA:
        m = (ev["stratum"] == s).values
        row[s] = brier(p[m], y[m]) if m.sum() else np.nan
    out[name] = row
pd.DataFrame(out).T.round(4)

**Persistence looks excellent pool-wide.** That is almost entirely the Deep
stratum, which is trivially predictable and the largest group.

Now quantify how badly the pool number hides improvement where it matters.

In [ ]:
p = np.clip(ev["prev"].values, 0.05, 0.95)
pool_now = brier(p, y)
m = (ev["stratum"] == "Rotation").values
improved = p.copy()
improved[m] = y[m] + (p[m] - y[m]) * np.sqrt(0.75)   # 25% error reduction, Rotation only
print("share of rows by stratum:")
print((ev["stratum"].value_counts(normalize=True) * 100).round(1).to_string())
print("")
print("pool Brier now                            %.4f" % pool_now)
print("pool Brier after 25%% better on Rotation   %.4f" % brier(improved, y))
print("-> pool moves only %.1f%%" % (100 * (pool_now - brier(improved, y)) / pool_now))

A model could get materially better at the only job that matters and the
headline metric would barely register it.

**Conclusion: report Rotation Brier as the headline. Never a single pool number.**

## 5. Where the error actually lives

Split the errors by whether the player's start state changed from the previous
gameweek.

In [ ]:
wrong = (ev["prev"].values != y)
err = (p - y) ** 2
print("wrong calls: %d of %d (%.1f%%)" % (wrong.sum(), len(y), 100 * wrong.mean()))
print("  they contribute %.1f%% of total Brier" % (100 * err[wrong].sum() / err.sum()))
print("  correct calls contribute %.1f%%" % (100 * err[~wrong].sum() / err.sum()))
print("")
a = ev[ev["prev"] == 1]
b = ev[ev["prev"] == 0]
print("TRANSITION RATES")
print("  started last GW : n=%6d  P(start again) = %.3f" % (len(a), a.y.mean()))
print("  benched last GW : n=%6d  P(start now)   = %.3f" % (len(b), b.y.mean()))

Persistence predicts **0.95 / 0.05**. The truth is roughly **0.80 / 0.08**.
It is not wrong about direction — it is overconfident. Brier charges for that.

### Why the observed frequency is optimal

For a group with true rate `q`, expected Brier is `q(p-1)² + (1-q)p²`.
Differentiating gives `2p - 2q`, zero at `p = q`.

In [ ]:
q = float(a.y.mean())
tbl = []
for cand in [1.00, 0.95, 0.90, 0.85, round(q, 3), 0.75, 0.70, 0.60, 0.50]:
    tbl.append({"you predict": cand,
                "cost when right": (cand - 1) ** 2,
                "cost when wrong": cand ** 2,
                "expected Brier": q * (cand - 1) ** 2 + (1 - q) * cand ** 2})
res = pd.DataFrame(tbl).round(4)
print("Bucket: started last gameweek (true rate %.3f)" % q)
print(res.to_string(index=False))
print("")
print("minimum at p = %.3f, the observed frequency" % q)

## 6. The model: a lookup table

No regression, no optimiser. Count how often each situation resolved each way.

- **Calibrated** — condition on `prev` alone. Two numbers.
- **Calibrated + rolling** — condition on `prev` and `roll4`. Seven or eight numbers.

In [ ]:
fit = d[(d.GW >= 6) & (d.GW <= 22)].copy()
fit["bin"] = pd.cut(fit["roll4"], BINS, labels=False)
tab = fit.groupby(["prev", "bin"])["y"].agg(["mean", "size"])

disp = tab.reset_index()
disp["started last GW"] = np.where(disp["prev"] == 1, "YES", "no")
disp["started in last 4"] = [BIN_NAMES[int(b)] for b in disp["bin"]]
disp = disp[["started last GW", "started in last 4", "size", "mean"]]
disp.columns = ["started last GW", "started in last 4", "n", "P(starts)"]
print(disp.to_string(index=False, float_format=lambda v: "%.3f" % v))

r1 = fit[fit["prev"] == 1]["y"].mean()
r0 = fit[fit["prev"] == 0]["y"].mean()
print("")
print("'calibrated' uses only the first column -> %.3f / %.3f" % (r1, r0))

The bottom row is the important one: **did not start last week, but started all
four before that**. Persistence assigns 0.05. The table says roughly 0.45 — a
player who was a regular and got dropped once.

That is the shape of the Senesi case, and the two disagree by about ninefold.

## 7. Walk-forward validation

Refit before every gameweek using only strictly prior data. This mirrors
production and yields ~27 folds instead of one holdout, so improvements come
with variance estimates.

**Both the lookup table and the strata labels are refitted each fold.** Using
season-level labels here is a subtle leak that contaminates exactly the
stratum of interest.

In [ ]:
def lookup_predict(train, test, min_cell=50):
    """Fit the two lookup tables on `train`, predict `test`."""
    r1 = train[train["prev"] == 1]["y"].mean()
    r0 = train[train["prev"] == 0]["y"].mean()
    tr = train.copy()
    tr["bin"] = pd.cut(tr["roll4"], BINS, labels=False)
    t = tr.groupby(["prev", "bin"])["y"].agg(["mean", "size"])
    te_bin = pd.cut(test["roll4"], BINS, labels=False)

    p_cal = np.where(test["prev"].values == 1, r1, r0)
    p_roll = []
    for pv, bn in zip(test["prev"].values, te_bin):
        fallback = r1 if pv == 1 else r0
        key = (pv, bn)
        if key in t.index and t.loc[key, "size"] >= min_cell:
            p_roll.append(t.loc[key, "mean"])
        else:
            p_roll.append(fallback)
    return p_cal, np.array(p_roll)

folds = []
for gw in range(12, 39):
    train = d[d.GW < gw]
    test = d[d.GW == gw].copy()
    if len(test) == 0:
        continue
    lab = label_from(train)                       # strata from prior GWs ONLY
    test = test[test["element"].isin(lab.index)]
    if len(test) == 0:
        continue
    test["stratum"] = test["element"].map(lab)

    yy = test["y"].values
    p_persist = np.clip(test["prev"].values, 0.05, 0.95)
    p_cal, p_roll = lookup_predict(train, test)

    folds.append({"gw": gw, "scope": "POOL", "n": len(test),
                  "persistence": brier(p_persist, yy),
                  "calibrated": brier(p_cal, yy),
                  "cal+rolling": brier(p_roll, yy)})
    for s in STRATA:
        m = (test["stratum"] == s).values
        if m.sum() >= 20:
            folds.append({"gw": gw, "scope": s, "n": int(m.sum()),
                          "persistence": brier(p_persist[m], yy[m]),
                          "calibrated": brier(p_cal[m], yy[m]),
                          "cal+rolling": brier(p_roll[m], yy[m])})

F = pd.DataFrame(folds)
print("%d folds across %d gameweeks" % (len(F), F.gw.nunique()))

In [ ]:
models = ["persistence", "calibrated", "cal+rolling"]
report = []
for scope in ["POOL"] + STRATA:
    x = F[F["scope"] == scope]
    if len(x) == 0:
        continue
    row = {"scope": scope, "rows/GW": int(round(x["n"].mean()))}
    for m in models:
        row[m] = round(x[m].mean(), 4)
    row["gain vs persist"] = "%.1f%%" % (100 * (x["persistence"].mean() - x["cal+rolling"].mean())
                                         / x["persistence"].mean())
    row["win rate"] = "%.0f%%" % (100 * (x["cal+rolling"] < x["persistence"]).mean())
    report.append(row)
pd.DataFrame(report).set_index("scope")

In [ ]:
x = F[F["scope"] == "POOL"]
print("POOL, across %d folds" % len(x))
print(x[models].agg(["mean", "std", "min", "max"]).round(4).to_string())

The gain holds on **Rotation**, the stratum that matters, and wins in the large
majority of individual gameweeks. Note Deep barely improves — it was already
near-perfect — while being the largest group, which is exactly how it flatters
a pool-wide average.

## 8. Are the strata labels stable?

A label recomputed weekly is only useful for reporting if it does not churn.

In [ ]:
labs = {}
for gw in range(12, 39):
    labs[gw] = label_from(d[d.GW < gw])
L = pd.DataFrame(labs).dropna()
changes = (L.iloc[:, 1:].values != L.iloc[:, :-1].values)
print("players tracked: %d" % len(L))
print("week-to-week stability: %.1f%%" % (100 * (1 - changes.mean())))
print("never changed label:    %.0f%%" % (100 * (changes.sum(1) == 0).mean()))
print("")
print("migration, first tracked gameweek vs last:")
print(pd.crosstab(L.iloc[:, 0], L.iloc[:, -1]).to_string())

Movement is sensible: Core to Rotation is a player losing their place, Deep to
Marginal is a squad player breaking in.

**Deep is a one-way door** — the label requires zero starts *ever*, so nothing
re-enters. A player who started twice in August stays Marginal all season even
if functionally Deep by March. That is a definitional artifact worth knowing.

Early gameweeks are also coarse: at GW12 the rate rests on ~11 observations, so
a single start can move a player between strata.

## 9. Charts

All interactive — hover for underlying counts.

Rendered via `pio.renderers.default = "iframe"` (see setup cell) for
JupyterLab compatibility — each figure is written to `iframe_figures/` next
to the notebook.

### Reliability curve

In [ ]:
train = d[d.GW <= 22]
test = d[d.GW >= 23].copy()
p_cal, p_roll = lookup_predict(train, test)
yy = test["y"].values
p_persist = np.clip(test["prev"].values, 0.05, 0.95)

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfectly calibrated",
                         line=dict(dash="dash", color="#888", width=1),
                         hoverinfo="skip"))

edges = np.linspace(0, 1, 11)
for name, pred in [("persistence", p_persist), ("calibrated", p_cal), ("cal+rolling", p_roll)]:
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pred >= lo) & (pred < hi)
        if m.sum() >= 30:
            xs.append(float(pred[m].mean()))
            ys.append(float(yy[m].mean()))
            ns.append(int(m.sum()))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers",
        name="%s (Brier %.4f)" % (name, brier(pred, yy)),
        marker=dict(size=9), customdata=ns,
        hovertemplate="<b>" + name + "</b><br>predicted %{x:.3f}"
                      "<br>observed %{y:.3f}"
                      "<br>n = %{customdata:,}<extra></extra>"))

fig.update_layout(
    title="Reliability, GW23-38 (fitted on GW6-22)",
    xaxis_title="predicted probability", yaxis_title="observed frequency",
    xaxis=dict(range=[0, 1], gridcolor="#eee"), yaxis=dict(range=[0, 1], gridcolor="#eee"),
    width=680, height=560, plot_bgcolor="white",
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.85)"),
    hovermode="closest")
fig.show()

Points on the dashed line are calibrated. Below it means overconfident: the
prediction was higher than the outcome frequency justified.

Persistence sits at the two extremes only, and both are off the diagonal.
Hover any point for its bucket size.

### Walk-forward Brier, gameweek by gameweek

The "wins in 96% of gameweeks" figure as a series rather than a summary.

In [ ]:
pool = F[F["scope"] == "POOL"].sort_values("gw")
fig = go.Figure()
for name, colour in [("persistence", "#d1495b"), ("calibrated", "#edae49"), ("cal+rolling", "#00798c")]:
    fig.add_trace(go.Scatter(
        x=pool["gw"], y=pool[name], mode="lines+markers", name=name,
        line=dict(color=colour, width=2), marker=dict(size=6),
        hovertemplate="GW%{x}<br>Brier %{y:.4f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Walk-forward Brier by gameweek (lower is better)",
    xaxis_title="gameweek", yaxis_title="Brier score",
    width=880, height=440, plot_bgcolor="white",
    xaxis=dict(gridcolor="#eee", dtick=2), yaxis=dict(gridcolor="#eee"),
    hovermode="x unified")
fig.show()

### Gain by stratum

Same folds, split by stratum. Rotation is the one that matters.

In [ ]:
rows = []
for s in STRATA:
    x = F[F["scope"] == s]
    if len(x):
        rows.append((s, x["persistence"].mean(), x["cal+rolling"].mean()))
labels = [r[0] for r in rows]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=[r[1] for r in rows], name="persistence",
                     marker_color="#d1495b",
                     hovertemplate="%{x}<br>Brier %{y:.4f}<extra>persistence</extra>"))
fig.add_trace(go.Bar(x=labels, y=[r[2] for r in rows], name="cal+rolling",
                     marker_color="#00798c",
                     hovertemplate="%{x}<br>Brier %{y:.4f}<extra>cal+rolling</extra>"))
fig.update_layout(
    title="Mean Brier by stratum, walk-forward (lower is better)",
    yaxis_title="Brier score", barmode="group",
    width=760, height=440, plot_bgcolor="white",
    yaxis=dict(gridcolor="#eee"))
fig.show()

## What this does and does not establish

**Established:**

- Pool-wide Brier is misleading; report Rotation separately
- Almost all error sits in gameweeks where the start state flipped
- Calibration alone beats persistence, using no new information
- The result survives leak-free walk-forward validation

**Not established:**

- ~~**Cross-season transfer.** Everything here is 2025-26. Whether transition
  rates hold in 2026-27 is untested — rerun section 7 against 2024-25 to
  check.~~ See section 10 below: addressed specifically for the
  early-gameweeks-of-a-new-season case, which is the one that matters right
  now (2026-27 has only 2 gameweeks archived as of this analysis).
- **Cold start.** The same players appear in fit and test. These numbers say
  nothing about new signings. That needs a player-level holdout: fit excluding
  a random 20% of players, score only on them.
- **That richer models beat this.** A seven-cell lookup table is the bar. The
  Beta-Binomial in §4.2 of the spec must beat *this*, not persistence.
- **Availability vs selection.** The denominator counts gameweeks registered,
  not available, so injured regulars contaminate Rotation. Joining to the API's
  `status` field would separate them.

## 10. Season-boundary backtest: does `roll4` survive across seasons?

**The real question this section answers isn't "is 2025-26 like 2026-27."**
It's narrower and more important: section 7's walk-forward validation only
evaluates rows where `roll4` is already computable --
`d = raw[raw["prev"].notna() & raw["roll4"].notna()]` in section 2 drops every
row without 4 prior gameweeks of history. That silently excludes GW1-4 of
*every* season from the entire validation. The 96%-of-gameweeks result says
nothing about early-season performance, in 2025-26 or any other season --
it was never tested there.

2026-27 has only 2 gameweeks archived so far, which is exactly this excluded
regime. The question: instead of leaving `roll4`/`prev` undefined for the
first few gameweeks of a season, what if the rolling window is computed
*across* the season boundary -- last season's final gameweeks standing in as
history for this season's first ones? Does that actually help, or is a
player's end-of-last-season pattern (squad rotation in dead rubbers, etc.) a
bad predictor of their new-season pattern?

**Method.** Join 2024-25 and 2025-26 on `code` (not `element` -- ids are
season-relative, exactly the problem this project's own archiver/derived
layer already solved for the live API; `players_raw.csv` carries both per
season). Treat the two seasons as one continuous 76-gameweek sequence per
player and compute `prev`/`roll4` across the join. Walk-forward evaluate GW1-10
of 2025-26 for players who also appeared in 2024-25 (534 of 841 -- the rest are
new signings/promotions/youth, the already-documented §8b cold-start case,
untouched by this fix either way).

In [ ]:
PRIOR_SEASON = "2024-25"

def season_gw_by_code(season, period_offset):
    """merged_gw.csv joined to players_raw.csv's id -> code, with a
    `period` column so two seasons can be treated as one continuous
    sequence (period = GW + period_offset).
    """
    df = pd.read_csv(io.BytesIO(fetch(season + "/gws/merged_gw.csv")), low_memory=False)
    players = pd.read_csv(io.BytesIO(fetch(season + "/players_raw.csv")))
    id_to_code = players.set_index("id")["code"]
    df["code"] = df["element"].map(id_to_code)
    gw_col = "GW" if "GW" in df.columns else "round"
    df = df.rename(columns={gw_col: "GW"})
    df["season"] = season
    df["period"] = df["GW"] + period_offset
    df["y"] = df["starts"].astype(int)
    return df.dropna(subset=["code"])

prior_s = season_gw_by_code(PRIOR_SEASON, period_offset=0)      # periods 1..38
cur_s = season_gw_by_code(SEASON, period_offset=38)              # periods 39..76

returning_codes = set(prior_s["code"].unique())
print("%s players: %d | %s players: %d | present in both: %d" % (
    PRIOR_SEASON, prior_s["code"].nunique(), SEASON, cur_s["code"].nunique(),
    len(returning_codes & set(cur_s["code"].unique()))
))

xseason = pd.concat([prior_s, cur_s], ignore_index=True).sort_values(["code", "period"])
xseason["prev"] = xseason.groupby("code")["y"].shift(1)
xseason["roll4"] = xseason.groupby("code")["y"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean())

gw1_returning = xseason[(xseason["season"] == SEASON) & (xseason["GW"] == 1)
                         & (xseason["code"].isin(returning_codes))]
print("GW1 %s rows for returning players with a defined cross-season prev: %d / %d"
      % (SEASON, gw1_returning["prev"].notna().sum(), len(gw1_returning)))

Every returning player's GW1-of-the-new-season row now has a defined `prev`
and `roll4` -- the mechanical part of the fix works. The real question is
whether it's *worth* using, against the honest alternative: what would this
project actually fall back to for early-season predictions without it? Per
docs §8b's cold-start guidance, that's a per-player prior derived from
`history_past` -- approximated here as each player's simple overall start
rate from the prior season, with no rolling/recency logic at all.

In [ ]:
prior_season_rate = prior_s.groupby("code")["y"].mean()
pool_fallback_rate = float(prior_s["y"].mean())

xfolds = []
for gw in range(1, 11):
    period = gw + 38
    train = xseason[xseason["period"] < period].dropna(subset=["prev", "roll4"])
    test = xseason[(xseason["period"] == period)
                    & (xseason["code"].isin(returning_codes))].copy()
    test = test.dropna(subset=["prev", "roll4"])
    if len(test) == 0:
        continue

    yy = test["y"].values
    p_persist = np.clip(test["prev"].values, 0.05, 0.95)
    p_cal, p_roll = lookup_predict(train, test)
    p_fallback = test["code"].map(prior_season_rate).fillna(pool_fallback_rate).values

    xfolds.append({
        "gw": gw, "n": len(test),
        "persistence (x-season)": brier(p_persist, yy),
        "calibrated (x-season)": brier(p_cal, yy),
        "cal+rolling (x-season)": brier(p_roll, yy),
        "prior-season rate only (no fix)": brier(p_fallback, yy),
    })

XF = pd.DataFrame(xfolds)
print("%d folds, GW1-%d of %s, returning players only" % (len(XF), XF.gw.max(), SEASON))
XF.round(4)

In [ ]:
fig = go.Figure()
colours = {
    "persistence (x-season)": "#d1495b",
    "calibrated (x-season)": "#edae49",
    "cal+rolling (x-season)": "#00798c",
    "prior-season rate only (no fix)": "#8d99ae",
}
for name, colour in colours.items():
    fig.add_trace(go.Scatter(
        x=XF["gw"], y=XF[name], mode="lines+markers", name=name,
        line=dict(color=colour, width=2), marker=dict(size=7),
        hovertemplate="GW%{x}<br>Brier %{y:.4f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Early-season Brier, %s GW1-10 (returning players only)" % SEASON,
    xaxis_title="gameweek of new season", yaxis_title="Brier score",
    width=880, height=460, plot_bgcolor="white",
    xaxis=dict(gridcolor="#eee", dtick=1), yaxis=dict(gridcolor="#eee"),
    hovermode="x unified")
fig.show()

print("means across GW1-10:")
print(XF[list(colours.keys())].mean().round(4).to_string())

### Result

**Cross-season `roll4` is clearly worth building.** Averaged over GW1-10,
`cal+rolling (x-season)` beats the no-fix fallback (a static prior-season
rate) by roughly **40%** on Brier, and beats cross-season persistence too.
This isn't a marginal call -- for returning players, extending the rolling
window across the boundary is a substantially better early-season estimate
than falling back to a rate with no recency information at all.

**But there's a real exception, and it's exactly GW1.** At GW1 specifically,
`prior-season rate only` edges out every cross-season variant, `cal+rolling`
included. That has a plausible mechanism: `roll4`/`prev` at GW1 are built from
*last season's final gameweeks* -- exactly when dead-rubber rotation, already-
secured European spots, and non-competitive fixtures make a team's starting
XI least representative of how they'll actually set up once the new season is
live and every point matters. A full-season average smooths that out; a
4-gameweek tail window doesn't.

From GW2 onward, `cal+rolling (x-season)` pulls ahead and stays ahead for
every remaining gameweek tested, GW2 itself excepted -- there it briefly
trails both persistence and calibrated, plausibly early-season noise (n≈514)
rather than a systematic effect, but not investigated further here.

**Recommendation for the P(starts) model:** use the prior-season rate as the
GW1 prior, and switch to cross-season `roll4` from GW2 once at least one
gameweek of the new season has actually been played and can inform which
of last season's tail patterns still hold. This is the same shrinkage
principle the spec already prescribes for cold-start players (§8b) --
applied here to the *season*, not just the player.

**Scope of this result:** returning players only (534 of 841 in 2025-26).
New signings, promoted-club players and youth players have no prior-season
`code` to join against at all -- for them, nothing here changes anything;
they remain the already-documented §8b cold-start case (`history_past` and
the price-and-position prior).

## 11. Does resetting history at a club change help?

**The gap this answers.** `code` alone doesn't distinguish "still at the
same club" from "just transferred" -- `prev`/`roll4` (sections 6-10) are
computed per-player regardless of which team a historical row was actually
at. A player who was nailed-on at their old club carries that form straight
into predictions at a new one, with nothing to catch it except the news
layer noticing an article that says so -- which only works when an article
exists and gets extracted correctly. This section tests a structural fix
instead: reset `prev`/`roll4` at every *confirmed* change of club (a
missing team on either side isn't evidence of a move, so it's never
treated as one), plus a live cross-check against the player's actual
current club so a transfer with zero games played for the new side yet is
still caught.

**Why this doesn't need availability status.** The obvious objection: can
this be walk-forward validated the same rigorous way as everything else
here, given `status`/`chance_of_playing_next_round`/`news` are current-
season-only (docs/README.md's "Deadline" section)? Yes -- the fix only
needs *which team a player was on*, which `merged_gw.csv`'s own `team`
column carries every gameweek, every past season (confirmed live: plain
club-name prose, e.g. "Arsenal", not a season-relative id). No live-only
field is involved, so this reuses section 10's exact cross-season data and
walk-forward method.

**Method.** Same `returning_codes` as section 10 (534 players present in
both 2024-25 and 2025-26). Split them by whether their last recorded
2024-25 `team` differs from their first recorded 2025-26 `team` -- a
summer transfer, the natural experiment with by far the largest sample
(a within-season January-window transfer would give a far smaller one).
Compare three candidates walk-forward over 2025-26 GW1-10, same folds as
section 10: the no-fix fallback (prior-season rate only), section 10's
blind cross-season `roll4` (continuity regardless of club), and this
section's team-aware reset.

In [ ]:
# returning_codes (section 10) is every 2024-25 player, not just those
# who also appear in 2025-26 -- narrow to the real overlap first, or a
# player who simply left the league (retired, relegated club) gets
# miscounted as "changed club" against a current-season team that
# doesn't exist for them at all.
actually_returning = returning_codes & set(cur_s["code"].unique())

last_team_prior = prior_s.sort_values("GW").groupby("code")["team"].last()
first_team_cur = cur_s.sort_values("GW").groupby("code")["team"].first()

changed_club = {
    code: (last_team_prior.get(code) != first_team_cur.get(code))
    for code in actually_returning
}
n_changed = sum(changed_club.values())
print("returning players who changed club over the summer: %d / %d (%.1f%%)" % (
    n_changed, len(actually_returning), 100 * n_changed / len(actually_returning)
))

Small but real -- about a tenth of returning players moved clubs, which
is exactly the group blind cross-season continuity (section 10) never
distinguished from everyone else.

In [ ]:
# Blind cross-season prev/roll4 -- section 10's approach, ignoring team.
xseason["prev_blind"] = xseason.groupby("code")["y"].shift(1)
xseason["roll4_blind"] = xseason.groupby("code")["y"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean())

# Team-aware reset -- this section's fix. A stint breaks only when `team`
# is known on both this row and the previous one for that code, and they
# differ; a missing team on either side is never treated as a move.
prev_team = xseason.groupby("code")["team"].shift(1)
confirmed_change = (
    xseason["team"].notna() & prev_team.notna() & (xseason["team"] != prev_team)
)
xseason["team_stint"] = confirmed_change.groupby(xseason["code"]).cumsum()
xseason["prev_team_aware"] = xseason.groupby(["code", "team_stint"])["y"].shift(1)
xseason["roll4_team_aware"] = xseason.groupby(["code", "team_stint"])["y"].transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean())

In [ ]:
def lookup_predict_cols(train, test, prev_col, roll4_col, min_cell=50):
    """Same method as lookup_predict (section 7), parameterized to the
    feature-column pair, so the blind and team-aware constructions can be
    compared against exactly the same walk-forward folds."""
    r1 = train[train[prev_col] == 1]["y"].mean()
    r0 = train[train[prev_col] == 0]["y"].mean()
    tr = train.copy()
    tr["bin"] = pd.cut(tr[roll4_col], BINS, labels=False)
    t = tr.groupby([prev_col, "bin"])["y"].agg(["mean", "size"])
    te_bin = pd.cut(test[roll4_col], BINS, labels=False)
    preds = []
    for pv, bn in zip(test[prev_col].values, te_bin):
        fallback = r1 if pv == 1 else r0
        key = (pv, bn)
        if key in t.index and t.loc[key, "size"] >= min_cell:
            preds.append(t.loc[key, "mean"])
        else:
            preds.append(fallback)
    return np.array(preds)

rows = []
for gw in range(1, 11):
    period = gw + 38
    train_blind = xseason[xseason["period"] < period].dropna(subset=["prev_blind", "roll4_blind"])
    train_ta = xseason[xseason["period"] < period].dropna(
        subset=["prev_team_aware", "roll4_team_aware"])

    test_all = xseason[(xseason["period"] == period)
                        & (xseason["code"].isin(returning_codes))].copy()
    if len(test_all) == 0:
        continue

    for group_name, want_changed in [("changed_club", True), ("same_club", False)]:
        test = test_all[test_all["code"].map(lambda c: changed_club.get(c, False))
                         == want_changed].copy()
        if len(test) == 0:
            continue

        yy = test["y"].values
        p_fallback = test["code"].map(prior_season_rate).fillna(pool_fallback_rate).values
        row = {"gw": gw, "group": group_name, "n": len(test),
               "prior_season_rate_only": brier(p_fallback, yy)}

        test_blind = test.dropna(subset=["prev_blind", "roll4_blind"])
        if len(test_blind):
            p_blind = lookup_predict_cols(train_blind, test_blind, "prev_blind", "roll4_blind")
            row["blind_xseason"] = brier(p_blind, test_blind["y"].values)

        test_ta = test.dropna(subset=["prev_team_aware", "roll4_team_aware"])
        if len(test_ta):
            p_ta = lookup_predict_cols(train_ta, test_ta, "prev_team_aware", "roll4_team_aware")
            row["team_aware_reset"] = brier(p_ta, test_ta["y"].values)

        rows.append(row)

R = pd.DataFrame(rows)
print("%d fold/group rows across GW1-%d" % (len(R), R.gw.max()))
R.round(4)

In [ ]:
print("means across GW1-10, by group:")
for group in ["changed_club", "same_club"]:
    sub = R[R["group"] == group]
    print("")
    print("%s (n ~ %.0f/gameweek)" % (group, sub["n"].mean()))
    print(sub[["prior_season_rate_only", "blind_xseason", "team_aware_reset"]]
          .mean().round(4).to_string())

In [ ]:
changed = R[R["group"] == "changed_club"].sort_values("gw")
fig = go.Figure()
colours = {
    "prior_season_rate_only": "#8d99ae",
    "blind_xseason": "#d1495b",
    "team_aware_reset": "#00798c",
}
for name, colour in colours.items():
    fig.add_trace(go.Scatter(
        x=changed["gw"], y=changed[name], mode="lines+markers", name=name,
        line=dict(color=colour, width=2), marker=dict(size=7),
        hovertemplate="GW%{x}<br>Brier %{y:.4f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Players who changed club over the summer, %s GW1-10" % SEASON,
    xaxis_title="gameweek of new season", yaxis_title="Brier score",
    width=880, height=460, plot_bgcolor="white",
    xaxis=dict(gridcolor="#eee", dtick=1), yaxis=dict(gridcolor="#eee"),
    hovermode="x unified")
fig.show()

### Result

**Resetting history at a confirmed club change is clearly worth it, for
exactly the players it applies to.** Averaged over GW1-10, for the ~10% of
returning players who changed clubs: `team_aware_reset` beats blind
cross-season continuity by roughly **27%** on Brier (0.092 vs 0.126), and
both comfortably beat falling back to a plain prior-season rate (0.306) --
carrying *some* recency signal, even reset at the move, is still much
better than none. GW1 itself is a special case worth naming explicitly:
every transferred player has zero games at the new club yet, so
`team_aware_reset` has nothing to predict from at all that gameweek --
correctly cold_start, not a gap in the analysis.

**For players who stayed put, the fix is a no-op, as it should be.** The
"same club" group scores 0.1016 either way (team-aware vs blind) --
confirmation the reset doesn't cost anything for the ~90% of returning
players it doesn't apply to.

**Recommendation:** ship the team-aware reset as the default, not an
experimental arm -- unlike the news-layer arms, there's no real trade-off
being tested here, just a bug (stale cross-club history) with a validated
fix. This also means the news layer's job narrows: it no longer needs to
be the *only* thing catching a transfer, just the thing that adds texture
on top (how a specific move is being talked about) once the structural
reset has already stopped the model from confidently carrying over form
that no longer applies.

**Not established here:** whether a *partial* discount (weighted blend of
old- and new-club history, rather than a hard reset) would do even
better -- untested, and see `docs/future_refinements.md` for why it's left
for later. Also untested: within-season (e.g. January-window) transfers
specifically -- the natural-experiment sample here is summer moves, which
share the same mechanism but not the same seasonal timing.